# Model Analysis — FP32 vs INT8

Compare the three exported model formats across:
- Model size (MB)
- Input/output shapes
- ONNX graph node count
- Layer-level weight statistics

**Prerequisites:** Run pipeline stages 1-3 first so artifacts exist.

In [ ]:
from pathlib import Path
import onnx
import onnxruntime as ort
import numpy as np
import pandas as pd

## 1. File Sizes

In [ ]:
artifacts = {
    'onnx_fp32':   Path('artifacts/model.onnx'),
    'onnx_int8':   Path('artifacts/model_int8.onnx'),
    'tflite_int8': Path('artifacts/model_int8.tflite'),
}

sizes = {
    name: round(path.stat().st_size / 1e6, 2)
    for name, path in artifacts.items()
    if path.exists()
}

df_size = pd.DataFrame(sizes.items(), columns=['format', 'size_mb'])
df_size['compression_ratio'] = df_size['size_mb'].iloc[0] / df_size['size_mb']
df_size

## 2. ONNX Graph Inspection

In [ ]:
def inspect_onnx(path: Path) -> dict:
    model = onnx.load(str(path))
    graph = model.graph
    return {
        'nodes':   len(graph.node),
        'inputs':  [(i.name, [d.dim_value for d in i.type.tensor_type.shape.dim]) for i in graph.input],
        'outputs': [(o.name, [d.dim_value for d in o.type.tensor_type.shape.dim]) for o in graph.output],
        'opset':   model.opset_import[0].version,
    }

for name, path in [('onnx_fp32', artifacts['onnx_fp32']), ('onnx_int8', artifacts['onnx_int8'])]:
    if path.exists():
        print(f"\n=== {name} ===")
        info = inspect_onnx(path)
        for k, v in info.items():
            print(f"  {k}: {v}")

## 3. ORT Session — Input / Output Metadata

In [ ]:
for name, path in [('onnx_fp32', artifacts['onnx_fp32']), ('onnx_int8', artifacts['onnx_int8'])]:
    if not path.exists():
        continue
    sess = ort.InferenceSession(str(path), providers=['CPUExecutionProvider'])
    print(f"\n=== {name} ===")
    for inp in sess.get_inputs():
        print(f"  INPUT  {inp.name}: shape={inp.shape}  type={inp.type}")
    for out in sess.get_outputs():
        print(f"  OUTPUT {out.name}: shape={out.shape}  type={out.type}")

## 4. Dummy Forward Pass

In [ ]:
dummy = np.random.rand(1, 3, 640, 640).astype(np.float32)

for name, path in [('onnx_fp32', artifacts['onnx_fp32']), ('onnx_int8', artifacts['onnx_int8'])]:
    if not path.exists():
        continue
    sess   = ort.InferenceSession(str(path), providers=['CPUExecutionProvider'])
    inp    = sess.get_inputs()[0].name
    out    = sess.run(None, {inp: dummy})
    print(f"{name}: output shape = {out[0].shape}")